In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from datetime import datetime
from torch.serialization import safe_globals, add_safe_globals
import shap
import numpy as np
from torchvision import transforms
import requests
import cv2
from PIL import Image
import io
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
import os
import kagglehub
import pydicom
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import time
import requests


# Custom Dataset class
class BrainCTDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            if img_path.lower().endswith('.dcm'):
                dcm = pydicom.dcmread(img_path)
                image = dcm.pixel_array.astype(float)
                window_center = 40
                window_width = 80
                min_value = window_center - window_width // 2
                max_value = window_center + window_width // 2
                image = np.clip(image, min_value, max_value)
                if len(image.shape) == 2:
                    image = np.stack([image] * 3, axis=-1)
            else:
                image = cv2.imread(img_path)
                if image is None:
                    raise ValueError(f"Failed to load image: {img_path}")
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
            image = image.astype(np.float32)
            if self.transform:
                image = self.transform(torch.from_numpy(image).permute(2, 0, 1))
            return image, label
        except Exception as e:
            print(f"Error processing image {img_path}: {str(e)}")
            return None, None

# Load dataset
def load_dataset(base_path, max_samples_per_class=100, random_seed=42):
    np.random.seed(random_seed)
    image_paths = []
    labels = []
    aneurysm_path = os.path.join(base_path, 'aneurysm')
    if os.path.exists(aneurysm_path):
        aneurysm_files = [os.path.join(aneurysm_path, f) for f in os.listdir(aneurysm_path)
                         if f.endswith(('.jpg', '.dcm'))]
        n_aneurysm = min(len(aneurysm_files), max_samples_per_class)
        selected_aneurysm = np.random.choice(aneurysm_files, n_aneurysm, replace=False)
        image_paths.extend(selected_aneurysm)
        labels.extend([1] * n_aneurysm)
    other_files = []
    for category in ['tumor', 'cancer']:
        category_path = os.path.join(base_path, category)
        if os.path.exists(category_path):
            other_files.extend([os.path.join(category_path, f) for f in os.listdir(category_path)
                               if f.endswith(('.jpg', '.dcm'))])
    n_other = min(len(other_files), max_samples_per_class)
    selected_other = np.random.choice(other_files, n_other, replace=False)
    image_paths.extend(selected_other)
    labels.extend([0] * n_other)
    return image_paths, labels

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
print("Loading dataset...")
dataset_path = kagglehub.dataset_download("trainingdatapro/computed-tomography-ct-of-the-brain")
base_path = os.path.join(dataset_path, 'files')
image_paths, labels = load_dataset(base_path, max_samples_per_class=100)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# Create datasets
train_dataset = BrainCTDataset(train_paths, train_labels, transform=transform)
test_dataset = BrainCTDataset(test_paths, test_labels, transform=transform)

# Create data loaders
batch_size = 64 if torch.cuda.is_available() else 16
num_workers = 4 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# 1. First, let's define a function to load the model properly
def load_model_safely(model_path, device):
    print(f"\nLoading model at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"User: krooldonutz")
    print(f"PyTorch version: {torch.__version__}")

    try:
        # Load with weights_only=False and safe globals
        add_safe_globals([torch.torch_version.TorchVersion])
        checkpoint = torch.load(model_path, weights_only=False, map_location=device)

        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                return checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                return checkpoint['state_dict']
            else:
                return checkpoint
        return checkpoint
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        raise

# 2. Update the model architecture to match the saved state
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        # Initialize VGG16 with pretrained weights
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Freeze feature layers
        for param in self.vgg16.features.parameters():
            param.requires_grad = False

        # Modify the classifier
        in_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.vgg16(x)

# 3. Load and initialize the model
try:
    # Initialize model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNNModel().to(device)

    # Load state dict
    model_path = 'my_model/model.pth'
    state_dict = load_model_safely(model_path, device)

    # Load state dict with strict=False to ignore missing keys
    incompatible_keys = model.load_state_dict(state_dict, strict=False)

    # Print information about loaded model
    print("\nModel loading results:")
    if incompatible_keys.missing_keys:
        print("\nMissing keys:")
        for key in incompatible_keys.missing_keys:
            print(f"- {key}")
    if incompatible_keys.unexpected_keys:
        print("\nUnexpected keys:")
        for key in incompatible_keys.unexpected_keys:
            print(f"- {key}")

    model.eval()
    print("\n✓ Model loaded and set to evaluation mode")

    # 4. Define evaluation function
    def evaluate_model(model, loader):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in loader:
                inputs, labels = batch
                if inputs is None:
                    continue
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total if total > 0 else 0
        return accuracy

    # 5. Evaluate model
    print("\nEvaluating model...")
    train_accuracy = evaluate_model(model, train_loader)
    test_accuracy = evaluate_model(model, test_loader)

    print(f"\nResults at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC:")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Test Accuracy: {test_accuracy:.2f}%")

except Exception as e:
    print(f"\n❌ Error: {str(e)}")
    import traceback
    print(traceback.format_exc())



# import shap
# import numpy as np
# import torch
# from torchvision import transforms
# import requests
# import cv2
# from datetime import datetime
# from PIL import Image
# import io
# import matplotlib.pyplot as plt
# import seaborn as sns

def fast_shap_analysis(model, image_url):
    try:
        start_time = datetime.utcnow()
        print(f"Analysis started at {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print(f"Analysis requested by: krooldonutz")
    
        # Image loading and preprocessing
        response = requests.get(image_url)
        if response.status_code != 200:
            return "Error: Could not download image from URL"

        image_pil = Image.open(io.BytesIO(response.content))
        image = np.array(image_pil)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
        image = image.astype(np.float32)

        # SHAP analysis
        masker = shap.maskers.Image("inpaint_telea", (224, 224, 3))

        def model_pipeline(x):
            x = torch.Tensor(x).permute(0, 3, 1, 2)
            normalize = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
            x = torch.stack([normalize(img) for img in x])
            with torch.no_grad():
                x = x.to(next(model.parameters()).device)
                output = model(x)
                probs = torch.nn.functional.softmax(output, dim=1)
            return probs.cpu().numpy()

        print("Analyzing the image with SHAP...")
        explainer = shap.Explainer(model=model_pipeline, masker=masker)
        shap_values = explainer(
            np.expand_dims(image, 0),
            max_evals=70,
            batch_size=5,
            outputs=shap.Explanation.argsort.flip[:1]
        )

        # Model prediction
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
        
        image_tensor = transform(Image.fromarray((image * 255).astype(np.uint8)))
        image_tensor = image_tensor.unsqueeze(0).to(next(model.parameters()).device)

        with torch.no_grad():
            output = model(image_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)
            pred = torch.argmax(probs, dim=1).item()
            conf = probs[0][pred].item()

        # Analysis calculations
        shap_abs = np.abs(shap_values.values)
        # Ensure proper reduction to 2D
        mean_shap = np.mean(shap_abs, axis=(0, -1))  # Average across batch and channels
        if len(mean_shap.shape) > 2:
            mean_shap = np.mean(mean_shap, axis=-1)  # Further reduce if needed
        
        # Calculate importance metrics
        importance_score = np.mean(np.abs(mean_shap))
        stability_score = 1 - (np.std(mean_shap) / (np.mean(mean_shap) + 1e-7))

        # Generate visualizations
        plt.figure(figsize=(15, 5))
        
        # Original image
        plt.subplot(1, 3, 1)
        plt.imshow(image)
        plt.title("Original Image")
        plt.axis('off')
        
        # SHAP heatmap
        plt.subplot(1, 3, 2)
        sns.heatmap(mean_shap, cmap='RdBu_r', center=0)
        plt.title("SHAP Importance Heatmap")
        plt.axis('off')
        
        # Overlay
        plt.subplot(1, 3, 3)
        plt.imshow(image)
        plt.imshow(mean_shap, cmap='RdBu_r', alpha=0.6)
        plt.title("Overlay of Image and SHAP Values")
        plt.axis('off')
        
        plt.tight_layout()
        plt.savefig('shap_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()

        # Analysis of quadrants
        h, w = mean_shap.shape
        quadrants = {
            'upper_left': mean_shap[:h//2, :w//2],
            'upper_right': mean_shap[:h//2, w//2:],
            'lower_left': mean_shap[h//2:, :w//2],
            'lower_right': mean_shap[h//2:, w//2:]
        }
        
        quadrant_scores = {k: float(np.mean(v)) for k, v in quadrants.items()}
        most_important_quadrant = max(quadrant_scores.items(), key=lambda x: x[1])[0]

        # Calculate attention metrics
        total_importance = float(np.sum(np.abs(mean_shap)))
        relative_importances = {k: float(np.sum(np.abs(v)))/total_importance 
                              for k, v in quadrants.items()}

        end_time = datetime.utcnow()
        analysis_duration = (end_time - start_time).total_seconds()

        # Generate report
        print("\n📋 Medical Image Analysis Report")
        print("=" * 50)
        
        print("\n🔍 Primary Findings:")
        diagnosis = 'detected an aneurysm' if pred == 1 else 'did not detect an aneurysm'
        print(f"• The model {diagnosis} in this image")
        print(f"• Confidence: {conf*100:.1f}%")
        
        confidence_desc = "high" if conf > 0.8 else "moderate" if conf > 0.6 else "low"
        print(f"• Confidence level: {confidence_desc}")

        print("\n🎯 Key Areas of Interest:")
        print(f"• Most significant region: {most_important_quadrant.replace('_', ' ')}")
        print(f"• Focus pattern: {'highly concentrated' if stability_score > 0.7 else 'distributed across multiple areas'}")
        
        print("\n💡 Detailed Regional Analysis:")
        for region, importance in relative_importances.items():
            region_name = region.replace('_', ' ').title()
            percentage = importance * 100
            attention_level = "High" if percentage > 30 else "Medium" if percentage > 20 else "Low"
            print(f"• {region_name}: {percentage:.1f}% ({attention_level} attention)")

        print("\n⚙️ Technical Metrics:")
        print(f"• Analysis time: {analysis_duration:.1f} seconds")
        print(f"• Model stability: {stability_score:.2f}")
        print(f"• Overall importance score: {importance_score:.4f}")
        
        print("\n📎 Additional Information:")
        print("• A detailed heatmap visualization has been saved as 'shap_analysis.png'")
        print(f"• Analysis completed at {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")

        return "Medical image analysis completed successfully with visualization"

    except Exception as e:
        import traceback
        print("Full error traceback:")
        print(traceback.format_exc())
        return f"Error in analysis: {str(e)}"


image_url = "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg"
result = fast_shap_analysis(model, image_url)
print(result)


In [ ]:
import boto3
import json

def test_lambda():
    # Initialize Lambda client
    lambda_client = boto3.client('lambda', region_name='ap-southeast-1')
    
    # Test event
    test_event = {
        "body": {
            "image_url": "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg",
            "user_id": "dc9a86e45e8845839e6bc638ea410da1"
        }
    }
    
    try:
        # Invoke Lambda function
        response = lambda_client.invoke(
            FunctionName='shap-analysis',
            InvocationType='RequestResponse',
            Payload=json.dumps(test_event)
        )
        
        # Read and parse the response
        response_payload = json.loads(response['Payload'].read())
        print("Status Code:", response['StatusCode'])
        print("Response:", json.dumps(response_payload, indent=2))
        
    except Exception as e:
        print("Error:", e)

if __name__ == "__main__":
    test_lambda()